# Spark Streaming Word Count from a Network Port (Google Colab)

**Goal:** count words arriving **live on a TCP port** using Spark Streaming's `socketTextStream`.

```
 ┌──────────────────────────┐   text lines over TCP   ┌──────────────────────────────────┐
 │ Python socket server     │ ──────────────────────▶ │ Spark Streaming                  │
 │ (background thread)      │    localhost:9999       │ socketTextStream -> word count   │
 │ sends 1 sentence / sec   │                         │ every 2-second micro-batch       │
 └──────────────────────────┘                         └──────────────────────────────────┘
```

**Why not `nc -lk 9999`?** On a laptop you would open a terminal, run netcat, and type sentences.
In Colab, `!nc -lk 9999` blocks the notebook so Spark could never start. Instead, we run a tiny socket server in a
**background thread** inside the notebook. Spark cannot tell the difference: it just connects to a port and reads lines.

**How to run:** `Runtime → Run all`. The streaming cell runs for about 30 seconds and stops on its own.

## Step 1 — Install Java and PySpark

In [ ]:
# ------------------------------------------------------------
# Install a supported Java (8/11/17) and PySpark 3.5.3 if needed
# ------------------------------------------------------------
import os, re, sys, subprocess
from importlib import metadata

def java_major_version():
    """Return the major Java version on PATH (e.g. 11, 17), or None if Java is missing."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        m = re.search(r'version "(\d+)(?:\.(\d+))?', out)
        major = int(m.group(1))
        return int(m.group(2)) if major == 1 else major      # "1.8" -> 8
    except Exception:
        return None

if java_major_version() not in (8, 11, 17):
    print("Installing OpenJDK 17 (about 30 seconds)...")
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

try:
    installed = metadata.version("pyspark")                # check WITHOUT importing pyspark
except metadata.PackageNotFoundError:
    installed = None
if installed != "3.5.3":
    print(f"Found pyspark={installed}; installing pyspark==3.5.3 ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"])

print("Java   :", java_major_version())
print("PySpark:", metadata.version("pyspark"))

Installing OpenJDK 17 (about 30 seconds)...


## Step 2 — Start Spark
A socket stream uses a **receiver**, which permanently occupies one thread. So Spark needs **at least 2 threads**:
one to receive data and one to process it. With `local[1]` the stream would receive data but never process it.

In [ ]:
import shutil, socket, threading, time
from operator import add

import pandas as pd
import matplotlib.pyplot as plt

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext

BASE_DIR = "/content/socket_wordcount" if os.path.isdir("/content") else os.path.abspath("socket_wordcount")
shutil.rmtree(BASE_DIR, ignore_errors=True)
os.makedirs(BASE_DIR, exist_ok=True)

conf = (SparkConf()
        .setMaster("local[3]")                                 # 3 threads: 1 receiver + 2 for processing
        .setAppName("SocketWordCount")
        .set("spark.default.parallelism", "2")
        .set("spark.ui.showConsoleProgress", "false")
        .set("spark.streaming.gracefulStopTimeout", "60s"))   # let the last batches finish on stop

spark = SparkSession.builder.config(conf=conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")                                        # hide INFO/WARN noise

# Needed for the running-total (stateful) and windowed counts in Step 4
sc.setCheckpointDir(os.path.join(BASE_DIR, "checkpoints"))

print("Spark", spark.version, "| master:", sc.master)

Spark 3.5.3 | master: local[3]


## Step 3 — Start a text server on a port (background thread)
The server waits for Spark to connect, then sends one sentence per second, each ending with a newline
(`socketTextStream` splits the stream into records on `\n`). Edit `TEXT_LINES` to send your own text.

In [ ]:
HOST = "localhost"
PREFERRED_PORT = 9999

TEXT_LINES = [
    "spark streaming makes real time processing easy",
    "real time data arrives as a continuous stream",
    "spark splits the stream into small batches",
    "each batch is processed like a normal spark job",
    "word count is the hello world of streaming",
    "streaming word count counts words in every batch",
    "windows count words over the last few seconds",
    "state keeps a running total for every word",
    "spark streaming reads text from a network port",
    "the receiver listens and spark counts the words",
    "data engineers love spark for big data",
    "this is the last line of the stream",
]
SEND_INTERVAL_SEC = 1.0


def pick_port(preferred):
    """Use the preferred port if it is free, otherwise let the OS give us a free one
    (useful when you re-run this cell and the old port is still in use)."""
    for port in (preferred, 0):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            s.bind((HOST, port))
            return s
        except OSError:
            s.close()
    raise RuntimeError("No free port available")


class LineServer(threading.Thread):
    """A tiny TCP server: accepts Spark's receiver and streams TEXT_LINES to it."""

    def __init__(self, lines, interval):
        super().__init__(daemon=True)                  # daemon: never blocks the notebook from exiting
        self.lines, self.interval = lines, interval
        self.stop_event = threading.Event()
        self.sent = 0
        self.server = pick_port(PREFERRED_PORT)
        self.port = self.server.getsockname()[1]
        self.server.listen(1)
        self.server.settimeout(1.0)                    # wake up every second to check stop_event

    def run(self):
        while not self.stop_event.is_set():
            try:
                conn, addr = self.server.accept()      # Spark's receiver connects here
            except socket.timeout:
                continue
            except OSError:
                break                                  # server socket closed
            print(f"[server] Spark receiver connected from {addr}")
            try:
                with conn:
                    for line in self.lines[self.sent:]:
                        if self.stop_event.is_set():
                            break
                        conn.sendall((line + "\n").encode("utf-8"))
                        self.sent += 1
                        print(f"[server] sent line {self.sent:>2}: {line}")
                        time.sleep(self.interval)
                    # Keep the connection open (idle) until we are told to stop,
                    # so the receiver does not keep reconnecting.
                    while not self.stop_event.is_set():
                        time.sleep(0.2)
            except (BrokenPipeError, ConnectionResetError):
                print("[server] receiver disconnected")  # happens when Spark stops first

    def stop(self):
        self.stop_event.set()
        self.server.close()


# Stop a server left over from a previous run of this cell
try:
    line_server.stop()
except NameError:
    pass

line_server = LineServer(TEXT_LINES, SEND_INTERVAL_SEC)
line_server.start()
PORT = line_server.port
print(f"Text server listening on {HOST}:{PORT} - waiting for Spark to connect")

## Step 4 — Word count on the port
Three results from the same stream:
1. **Per-batch word count** — `flatMap → map → reduceByKey` (forgets after each batch)
2. **Windowed word count** — last 6 seconds, updated every 2 seconds (`reduceByKeyAndWindow`)
3. **Running total** — counts since the stream started (`updateStateByKey`)

In [ ]:
BATCH_INTERVAL = 2                         # seconds per micro-batch
RUN_SECONDS = 30                           # stream for 30 s, then stop automatically

ssc = StreamingContext(sc, BATCH_INTERVAL)

# 1) Input DStream: connect to the port. Each line received becomes one record.
lines = ssc.socketTextStream(HOST, PORT)

# 2) Classic word count (per batch)
words = (lines
         .flatMap(lambda line: line.lower().split())        # line -> words
         .filter(lambda w: w.isalpha()))                     # keep only real words
pairs = words.map(lambda w: (w, 1))                          # word -> (word, 1)
batch_counts = pairs.reduceByKey(add)                        # sum within this batch

# 3) Windowed count: last 6 s, sliding every 2 s (inverse function = incremental update)
window_counts = pairs.reduceByKeyAndWindow(add, lambda a, b: a - b,
                                           windowDuration=6, slideDuration=2,
                                           filterFunc=lambda kv: kv[1] > 0)

# 4) Running total since start (stateful)
def update_total(new_counts, running_total):
    return sum(new_counts) + (running_total or 0)

total_counts = pairs.updateStateByKey(update_total)

# ---------------- Output operations ----------------
batch_log = []                                               # results kept for analysis after the run
final_totals = {}

def show_batch(batch_time, rdd):
    """Runs on the driver every batch: print the top words of this batch."""
    top = rdd.takeOrdered(5, key=lambda kv: (-kv[1], kv[0]))
    n_words = rdd.map(lambda kv: kv[1]).sum() if not rdd.isEmpty() else 0
    batch_log.append({"batch_time": batch_time.strftime("%H:%M:%S"), "words": n_words})
    print(f"\n[{batch_time.strftime('%H:%M:%S')}] batch words={n_words:<3} top: {top}")

def show_window(batch_time, rdd):
    top = rdd.takeOrdered(3, key=lambda kv: (-kv[1], kv[0]))
    print(f"            last 6s top: {top}")

def save_totals(batch_time, rdd):
    final_totals.clear()
    final_totals.update(dict(rdd.collect()))                 # latest snapshot of the running totals

batch_counts.foreachRDD(show_batch)
window_counts.foreachRDD(show_window)
total_counts.foreachRDD(save_totals)

# ---------------- Run ----------------
print(f"Starting stream on {HOST}:{PORT} for {RUN_SECONDS} seconds...")
ssc.start()
ssc.awaitTerminationOrTimeout(RUN_SECONDS)
ssc.stop(stopSparkContext=False, stopGraceFully=True)        # keep Spark alive for the next cells
line_server.stop()
print(f"\n✅ Stream stopped. Server sent {line_server.sent} of {len(TEXT_LINES)} lines.")

## Step 5 — Results

In [ ]:
totals_df = (pd.DataFrame(sorted(final_totals.items(), key=lambda kv: (-kv[1], kv[0])),
                          columns=["word", "count"]))
print("Total distinct words:", len(totals_df), "| total words:", int(totals_df["count"].sum()) if len(totals_df) else 0)
display(totals_df.head(15))

# Cross-check: the same count computed without Spark on the lines that were sent
expected = {}
for line in TEXT_LINES[:line_server.sent]:
    for w in line.lower().split():
        if w.isalpha():
            expected[w] = expected.get(w, 0) + 1
print("Spark totals match plain-Python count:", final_totals == expected)

if len(totals_df):
    top = totals_df.head(10)
    plt.figure(figsize=(10, 4))
    plt.bar(top["word"], top["count"])
    plt.title("Top 10 words (running total from the socket stream)")
    plt.ylabel("Count")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

### What to notice
* Each 2-second batch holds about 2 sentences, because the server sends one per second.
* **Per-batch** counts reset every batch, the **6-second window** holds roughly the last 6 sentences,
  and the **running total** keeps growing until the stream stops.
* After the server runs out of lines, batches become empty but the stream keeps ticking until we stop it.

### Try it yourself
1. Change `TEXT_LINES` (or add more lines) and re-run Steps 3 and 4.
2. Change `BATCH_INTERVAL` to 1 or 5 and see how the batch sizes change.
3. Set `.setMaster("local[1]")` in Step 2 (restart the session first): data is received but never processed.
   That shows why a receiver needs its own thread.

In [ ]:
# ------------------------------------------------------------
# Clean up: stop the server and Spark
# ------------------------------------------------------------
try:
    line_server.stop()
except NameError:
    pass
active = StreamingContext.getActive()
if active is not None:
    active.stop(stopSparkContext=False, stopGraceFully=False)
spark.stop()
print("Stopped. 🎉")